# Proyek UAS Mata Kuliah Kecerdasan Buatan Training dataset Bahasa Isyarat menggunakan CNN+LSTM
Kelompok Rabu-4:
- Nama (NPM) 
- Nama (NPM) 
- Nama (NPM) 
- Nama (NPM) 

# 1. Import Dependency

In [1]:
import numpy as np
from matplotlib import pyplot as plt
import sklearn
import tensorflow as tf
import os
import cv2
import time

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision


I0000 00:00:1779118868.636464   10724 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## 1.1. Setting Tensorflow GPU

In [3]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


# 2. Setup Open CV dan Mediapipe HandLanmarker

## 2.1. Inisialisasi Mediapipe HandLandmarker

In [8]:
BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

In [9]:
options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='handlandmarker/hand_landmarker.task',
                             delegate=BaseOptions.Delegate.GPU),
    running_mode=VisionRunningMode.IMAGE,
    num_hands=2) 

In [ ]:
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    results = model.detect(image)
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    return image, results

## 2.2. Inisialisasi OpenCV 2

In [10]:
cap = cv2.VideoCapture(0, cv2.CAP_V4L2) # Setting Capture untuk di linux

if not cap.isOpened():
    print("Cannot open camera")
else:
    while cap.isOpened():

        # membaca feed
        ret, frame = cap.read()
        
        if not ret or frame is None:
            print("Camera/Frame error.")
            break

        # show feed
        cv2.imshow('OpenCV Feed', frame)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break
cap.release()
cv2.destroyAllWindows()


# 3. Import Dataset

In [ ]:
DATA_PATH = os.path.join('dataset')

# 4. Preprocessing Dataset

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Conv1D, MaxPooling1D
from tensorflow.keras.callbacks import TensorBoard, EarlyStopping

In [ ]:
video_frames = 60

In [ ]:
actions = np.array([name for name in os.listdir(DATA_PATH) if os.path.isdir(os.path.join(DATA_PATH, name))])

label_map = {label:num for num, label in enumerate(actions)}

In [ ]:
print("Daftar actions:", actions)
print("Label map:", label_map)

In [ ]:
sequences, labels = [], []
for action in actions:
    action_path = os.path.join(DATA_PATH, action)
    
    # Mengambil otomatis nama list folder video (sequence) yang ada di dalam masing-masing action
    video_sequences = [d for d in os.listdir(action_path) if os.path.isdir(os.path.join(action_path, d))]
    
    for sequence in video_sequences:
        window = []
        for frame_num in range(video_frames):
            res = np.load(os.path.join(DATA_PATH, action, str(sequence), "{}.npy".format(frame_num)))
            window.append(res)
    
    sequences.append(window)
    labels.append(label_map[action])

In [ ]:
x = np.array(sequences)
y = to_categorical(labels).astype(int)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2)
print("Shape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)

# 5. Training Base Model

In [ ]:
# 1. Definisikan Arsitektur CNN + LSTM
model = Sequential()

# CNN Layers (untuk mengekstrak fitur spasial dari data urutan keypoint)
# WAJIB DISESUAIKAN: input_shape (jumlah_frame_per_video, jumlah_features_keypoints)
# Contoh ini memakai asusmsi 25 sequence frame, dengan 63 titik data matriks Mediapipe
model.add(Conv1D(filters=64, kernel_size=3, activation='relu', padding='same', input_shape=(60, 126)))
model.add(MaxPooling1D(pool_size=2))
model.add(Conv1D(filters=128, kernel_size=3, activation='relu', padding='same'))
model.add(MaxPooling1D(pool_size=2))

# LSTM Layers (untuk mempelajari Temporal/Sequence pergerakan titik waktu)
model.add(LSTM(64, return_sequences=True, activation='relu'))
model.add(Dropout(0.2))
model.add(LSTM(128, return_sequences=False, activation='relu'))

# Dense Layers (Klasifikasi akhir)
num_classes = len(actions) # WAJIB DISESUAIKAN: Ganti dengan total label dari class/gerakan sign language anda.
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(num_classes, activation='softmax'))

# Compile Model
model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['categorical_accuracy'])
model.summary()

In [ ]:
log_dir = os.path.join('Logs')
os.makedirs(log_dir, exist_ok=True)

tb_callback = TensorBoard(log_dir=log_dir)
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

history = model.fit(
    X_train, 
    y_train, 
    validation_data=(X_test, y_test), 
    epochs=200, 
    callbacks=[tb_callback, early_stopping]
)